# Advanced Coroutine Pipelines: Pushing Data with `send`

This notebook develops **push-based data pipelines** using Python generator coroutines.

You will build and test reusable stages for:

- mapping, filtering, tapping, and collecting data;
- graceful shutdown and downstream cleanup;
- fan-out, routing, batching, rolling windows, and stateful aggregation;
- error handling, dead-letter queues, instrumentation, and dynamic reconfiguration;
- reasoning about limitations such as synchronous blocking and lack of backpressure.

Each problem contains:

1. a specification;
2. design constraints;
3. a complete solution;
4. executable tests.

> Run the notebook from top to bottom. All examples use the Python standard library only.


## 0. Core Utilities and Best Practices

A generator coroutine must be advanced to its first `yield` before `.send(value)` can be used.  
The decorator below auto-primes the coroutine and preserves the wrapped function's metadata.

Best practices used throughout this notebook:

- use `functools.wraps`;
- keep stages small and composable;
- pass downstream targets explicitly;
- propagate `.close()` when a stage owns its downstream target;
- use `try/finally` for cleanup;
- test stages with in-memory collectors instead of relying only on `print`;
- make control messages explicit with unique sentinel objects;
- avoid silently swallowing unexpected exceptions;
- document whether a stage transforms, filters, routes, buffers, or terminates data.


In [1]:
from __future__ import annotations

from collections import defaultdict, deque
from dataclasses import dataclass
from functools import wraps
from statistics import mean
from typing import Any, Callable, Deque, Dict, Generator, Hashable, Iterable, List, Mapping, MutableMapping, Optional, Sequence, Tuple


def coroutine(func: Callable[..., Generator[Any, Any, Any]]) -> Callable[..., Generator[Any, Any, Any]]:
    """Decorator that creates and automatically primes a generator coroutine."""
    @wraps(func)
    def start(*args: Any, **kwargs: Any) -> Generator[Any, Any, Any]:
        gen = func(*args, **kwargs)
        next(gen)
        return gen
    return start


def close_quietly(*targets: Any) -> None:
    """Close every target that exposes a callable close method."""
    for target in targets:
        close = getattr(target, "close", None)
        if callable(close):
            try:
                close()
            except RuntimeError:
                # A generator may already be executing or closed.
                pass


FLUSH = object()
STOP = object()


### A test-friendly sink

A collector makes pipeline behavior easy to verify with assertions.


In [2]:
@coroutine
def collector(storage: List[Any]) -> Generator[None, Any, None]:
    """Append every received item to storage."""
    while True:
        item = yield
        storage.append(item)


@coroutine
def printer(prefix: str = "") -> Generator[None, Any, None]:
    """Print every received item."""
    while True:
        item = yield
        print(f"{prefix}{item}")


## Problem 1 — Build a Reusable Map → Filter → Sink Pipeline

### Task

Implement reusable stages equivalent to functional `map` and `filter`.

Construct a pipeline that:

1. squares each integer;
2. keeps only results divisible by `3`;
3. stores results in a list.

### Design constraints

- Each stage must forward data with `.send`.
- The stage must close its downstream target when it is closed.
- The solution must be generic: transformation and predicate logic are passed as callables.


In [3]:
@coroutine
def map_stage(
    transform: Callable[[Any], Any],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Transform each item and forward the result."""
    try:
        while True:
            item = yield
            target.send(transform(item))
    finally:
        close_quietly(target)


@coroutine
def filter_stage(
    predicate: Callable[[Any], bool],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Forward only items for which predicate(item) is truthy."""
    try:
        while True:
            item = yield
            if predicate(item):
                target.send(item)
    finally:
        close_quietly(target)


In [4]:
problem1_results: List[int] = []

problem1_pipeline = map_stage(
    lambda x: x * x,
    filter_stage(
        lambda squared: squared % 3 == 0,
        collector(problem1_results),
    ),
)

for value in range(1, 11):
    problem1_pipeline.send(value)

problem1_pipeline.close()

assert problem1_results == [9, 36, 81]
problem1_results


[9, 36, 81]

### Why the stage order matters

The data flow is:

```text
input → square → divisible-by-3 filter → collector
```

Changing the order can change semantics. Filtering original numbers divisible by `3` and then squaring happens to produce the same result here, but that equivalence does not hold for arbitrary transforms and predicates.


## Problem 2 — Add an Observable `tap` Stage Without Changing Data

### Task

Create a `tap_stage` that performs a side effect and forwards the original item unchanged.

Use it to capture an audit trail between a normalization stage and a validation stage.

Input records:

```python
{"sensor": "A", "temperature": " 21.5 "}
{"sensor": "B", "temperature": "bad"}
{"sensor": "C", "temperature": "105"}
```

Requirements:

- normalize valid temperature strings to `float`;
- reject malformed records;
- retain only temperatures in the inclusive interval `[-50, 100]`;
- preserve an audit copy of every successfully normalized record.


In [5]:
@coroutine
def tap_stage(
    action: Callable[[Any], None],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Perform a side effect, then forward the original item unchanged."""
    try:
        while True:
            item = yield
            action(item)
            target.send(item)
    finally:
        close_quietly(target)


def normalize_temperature(record: Mapping[str, Any]) -> Dict[str, Any]:
    """Return a normalized copy or raise ValueError/KeyError."""
    normalized = dict(record)
    normalized["temperature"] = float(str(record["temperature"]).strip())
    return normalized


A robust pipeline should decide deliberately what to do with malformed data.  
The next stage sends bad records to a separate error target instead of crashing the whole stream.


In [6]:
@dataclass(frozen=True)
class RejectedItem:
    item: Any
    error_type: str
    message: str


@coroutine
def try_map_stage(
    transform: Callable[[Any], Any],
    target: Generator[Any, Any, Any],
    error_target: Generator[Any, Any, Any],
    handled_exceptions: Tuple[type[BaseException], ...] = (ValueError, TypeError, KeyError),
) -> Generator[None, Any, None]:
    """Transform items; route expected data errors to error_target."""
    try:
        while True:
            item = yield
            try:
                target.send(transform(item))
            except handled_exceptions as exc:
                error_target.send(
                    RejectedItem(
                        item=item,
                        error_type=type(exc).__name__,
                        message=str(exc),
                    )
                )
    finally:
        close_quietly(target, error_target)


In [7]:
records = [
    {"sensor": "A", "temperature": " 21.5 "},
    {"sensor": "B", "temperature": "bad"},
    {"sensor": "C", "temperature": "105"},
]

audit_log: List[Dict[str, Any]] = []
accepted_records: List[Dict[str, Any]] = []
rejected_records: List[RejectedItem] = []

problem2_pipeline = try_map_stage(
    normalize_temperature,
    tap_stage(
        lambda record: audit_log.append(dict(record)),
        filter_stage(
            lambda record: -50 <= record["temperature"] <= 100,
            collector(accepted_records),
        ),
    ),
    collector(rejected_records),
)

for record in records:
    problem2_pipeline.send(record)

problem2_pipeline.close()

assert audit_log == [
    {"sensor": "A", "temperature": 21.5},
    {"sensor": "C", "temperature": 105.0},
]
assert accepted_records == [{"sensor": "A", "temperature": 21.5}]
assert len(rejected_records) == 1
assert rejected_records[0].error_type == "ValueError"

accepted_records, rejected_records


([{'sensor': 'A', 'temperature': 21.5}],
 [RejectedItem(item={'sensor': 'B', 'temperature': 'bad'}, error_type='ValueError', message="could not convert string to float: 'bad'")])

## Problem 3 — Fan-Out: Broadcast One Stream to Multiple Branches

### Task

Create a stage that sends each item to multiple targets.

Use it to process transaction amounts in two independent branches:

- branch A stores every amount;
- branch B stores only amounts of at least `100`, converted to strings such as `"HIGH: 250.00"`.

### Important design question

A broadcast is synchronous. A slow branch delays every other branch because `.send` calls happen one after another.


In [8]:
@coroutine
def broadcast(
    *targets: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Send each item to every target in declaration order."""
    try:
        while True:
            item = yield
            for target in targets:
                target.send(item)
    finally:
        close_quietly(*targets)


In [9]:
all_amounts: List[float] = []
high_value_labels: List[str] = []

problem3_pipeline = broadcast(
    collector(all_amounts),
    filter_stage(
        lambda amount: amount >= 100,
        map_stage(
            lambda amount: f"HIGH: {amount:.2f}",
            collector(high_value_labels),
        ),
    ),
)

for amount in [12.5, 250, 99.99, 100, 1000.5]:
    problem3_pipeline.send(amount)

problem3_pipeline.close()

assert all_amounts == [12.5, 250, 99.99, 100, 1000.5]
assert high_value_labels == [
    "HIGH: 250.00",
    "HIGH: 100.00",
    "HIGH: 1000.50",
]

all_amounts, high_value_labels


([12.5, 250, 99.99, 100, 1000.5],
 ['HIGH: 250.00', 'HIGH: 100.00', 'HIGH: 1000.50'])

## Problem 4 — Route Events by Key

### Task

Implement a router that chooses one target according to a key function.

Requirements:

- route events by their `"level"` field;
- support `"INFO"`, `"WARNING"`, and `"ERROR"`;
- send unknown levels to a default target;
- close each unique target exactly once when the router closes.


In [10]:
@coroutine
def route_by_key(
    key_func: Callable[[Any], Hashable],
    routes: Mapping[Hashable, Generator[Any, Any, Any]],
    default_target: Optional[Generator[Any, Any, Any]] = None,
) -> Generator[None, Any, None]:
    """Route each item to a target selected by key_func(item)."""
    unique_targets = list({id(target): target for target in routes.values()}.values())
    if default_target is not None and all(default_target is not t for t in unique_targets):
        unique_targets.append(default_target)

    try:
        while True:
            item = yield
            key = key_func(item)
            target = routes.get(key, default_target)
            if target is None:
                raise KeyError(f"No route configured for key {key!r}")
            target.send(item)
    finally:
        close_quietly(*unique_targets)


In [11]:
info_events: List[Dict[str, Any]] = []
warning_events: List[Dict[str, Any]] = []
error_events: List[Dict[str, Any]] = []
unknown_events: List[Dict[str, Any]] = []

problem4_pipeline = route_by_key(
    key_func=lambda event: event["level"],
    routes={
        "INFO": collector(info_events),
        "WARNING": collector(warning_events),
        "ERROR": collector(error_events),
    },
    default_target=collector(unknown_events),
)

events = [
    {"level": "INFO", "message": "started"},
    {"level": "ERROR", "message": "disk full"},
    {"level": "DEBUG", "message": "cache miss"},
    {"level": "WARNING", "message": "high memory"},
]

for event in events:
    problem4_pipeline.send(event)

problem4_pipeline.close()

assert [e["message"] for e in info_events] == ["started"]
assert [e["message"] for e in warning_events] == ["high memory"]
assert [e["message"] for e in error_events] == ["disk full"]
assert [e["message"] for e in unknown_events] == ["cache miss"]

info_events, warning_events, error_events, unknown_events


([{'level': 'INFO', 'message': 'started'}],
 [{'level': 'WARNING', 'message': 'high memory'}],
 [{'level': 'ERROR', 'message': 'disk full'}],
 [{'level': 'DEBUG', 'message': 'cache miss'}])

## Problem 5 — Fixed-Size Batching with Explicit Flush

### Task

Create a batching stage that groups items into lists of size `n`.

Requirements:

- emit a batch immediately when it reaches `n`;
- emit a partial batch when receiving the sentinel `FLUSH`;
- emit any remaining partial batch during `.close()`;
- reject invalid batch sizes;
- never forward the mutable internal buffer directly.

Copying the buffer before sending prevents downstream code from accidentally mutating internal state.


In [12]:
@coroutine
def batch_stage(
    size: int,
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Group items into lists and support explicit or close-time flushing."""
    if size <= 0:
        raise ValueError("size must be a positive integer")

    buffer: List[Any] = []

    def emit() -> None:
        if buffer:
            target.send(list(buffer))
            buffer.clear()

    try:
        while True:
            item = yield
            if item is FLUSH:
                emit()
                continue

            buffer.append(item)
            if len(buffer) >= size:
                emit()
    finally:
        emit()
        close_quietly(target)


In [13]:
batches: List[List[int]] = []
problem5_pipeline = batch_stage(3, collector(batches))

for value in [1, 2, 3, 4, 5]:
    problem5_pipeline.send(value)

problem5_pipeline.send(FLUSH)

for value in [6, 7, 8, 9]:
    problem5_pipeline.send(value)

problem5_pipeline.close()

assert batches == [[1, 2, 3], [4, 5], [6, 7, 8], [9]]
batches


[[1, 2, 3], [4, 5], [6, 7, 8], [9]]

## Problem 6 — Rolling Window Average

### Task

Create a stage that maintains the most recent `window_size` numeric values.

For each received value, send a record containing:

- the original value;
- the current window contents;
- the current average;
- a Boolean indicating whether the window is full.

The stage should start producing output immediately, even before the window becomes full.


In [14]:
@dataclass(frozen=True)
class RollingAverage:
    value: float
    window: Tuple[float, ...]
    average: float
    is_full: bool


@coroutine
def rolling_average_stage(
    window_size: int,
    target: Generator[Any, Any, Any],
) -> Generator[None, float, None]:
    """Emit a rolling-average record for every numeric input."""
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    window: Deque[float] = deque(maxlen=window_size)

    try:
        while True:
            value = float((yield))
            window.append(value)
            target.send(
                RollingAverage(
                    value=value,
                    window=tuple(window),
                    average=mean(window),
                    is_full=len(window) == window_size,
                )
            )
    finally:
        close_quietly(target)


In [15]:
rolling_results: List[RollingAverage] = []
problem6_pipeline = rolling_average_stage(3, collector(rolling_results))

for value in [10, 20, 30, 60]:
    problem6_pipeline.send(value)

problem6_pipeline.close()

assert [round(item.average, 2) for item in rolling_results] == [10.0, 15.0, 20.0, 36.67]
assert rolling_results[-1].window == (20.0, 30.0, 60.0)
assert rolling_results[-1].is_full is True

rolling_results


[RollingAverage(value=10.0, window=(10.0,), average=10.0, is_full=False),
 RollingAverage(value=20.0, window=(10.0, 20.0), average=15.0, is_full=False),
 RollingAverage(value=30.0, window=(10.0, 20.0, 30.0), average=20.0, is_full=True),
 RollingAverage(value=60.0, window=(20.0, 30.0, 60.0), average=36.666666666666664, is_full=True)]

## Problem 7 — Stateful Grouped Aggregation

### Task

Maintain a running total and count for each category.

For every input pair `(category, amount)`, emit the updated state for that category.

Example:

```text
("books", 10) → count=1, total=10, average=10
("books", 20) → count=2, total=30, average=15
```

The solution should not leak a mutable internal dictionary to downstream stages.


In [16]:
@dataclass(frozen=True)
class GroupStats:
    key: Hashable
    count: int
    total: float
    average: float


@coroutine
def grouped_stats_stage(
    target: Generator[Any, Any, Any],
) -> Generator[None, Tuple[Hashable, float], None]:
    """Emit updated count, total, and average for each key."""
    counts: MutableMapping[Hashable, int] = defaultdict(int)
    totals: MutableMapping[Hashable, float] = defaultdict(float)

    try:
        while True:
            key, raw_amount = yield
            amount = float(raw_amount)

            counts[key] += 1
            totals[key] += amount

            target.send(
                GroupStats(
                    key=key,
                    count=counts[key],
                    total=totals[key],
                    average=totals[key] / counts[key],
                )
            )
    finally:
        close_quietly(target)


In [17]:
stats_updates: List[GroupStats] = []
problem7_pipeline = grouped_stats_stage(collector(stats_updates))

for item in [
    ("books", 10),
    ("games", 50),
    ("books", 20),
    ("games", 30),
    ("books", 15),
]:
    problem7_pipeline.send(item)

problem7_pipeline.close()

assert stats_updates[-1] == GroupStats(
    key="books",
    count=3,
    total=45.0,
    average=15.0,
)

stats_updates


[GroupStats(key='books', count=1, total=10.0, average=10.0),
 GroupStats(key='games', count=1, total=50.0, average=50.0),
 GroupStats(key='books', count=2, total=30.0, average=15.0),
 GroupStats(key='games', count=2, total=80.0, average=40.0),
 GroupStats(key='books', count=3, total=45.0, average=15.0)]

## Problem 8 — Recoverable Errors and a Dead-Letter Queue

### Task

Build a data-quality pipeline for raw integers represented as strings.

Pipeline behavior:

1. parse each item with `int`;
2. reject malformed values into a dead-letter queue;
3. reject negative parsed integers into a second validation queue;
4. square accepted values;
5. collect final results.

Expected input:

```python
["10", "-2", "hello", "7", "3.5", "0"]
```


In [18]:
@dataclass(frozen=True)
class ValidationFailure:
    item: Any
    reason: str


@coroutine
def validate_stage(
    predicate: Callable[[Any], bool],
    reason: str,
    target: Generator[Any, Any, Any],
    rejected_target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Forward valid items and route invalid items to rejected_target."""
    try:
        while True:
            item = yield
            if predicate(item):
                target.send(item)
            else:
                rejected_target.send(ValidationFailure(item=item, reason=reason))
    finally:
        close_quietly(target, rejected_target)


In [19]:
parse_failures: List[RejectedItem] = []
validation_failures: List[ValidationFailure] = []
squared_values: List[int] = []

problem8_pipeline = try_map_stage(
    int,
    validate_stage(
        predicate=lambda value: value >= 0,
        reason="value must be non-negative",
        target=map_stage(lambda value: value * value, collector(squared_values)),
        rejected_target=collector(validation_failures),
    ),
    error_target=collector(parse_failures),
    handled_exceptions=(ValueError, TypeError),
)

for raw in ["10", "-2", "hello", "7", "3.5", "0"]:
    problem8_pipeline.send(raw)

problem8_pipeline.close()

assert squared_values == [100, 49, 0]
assert [failure.item for failure in parse_failures] == ["hello", "3.5"]
assert [failure.item for failure in validation_failures] == [-2]

squared_values, parse_failures, validation_failures


([100, 49, 0],
 [RejectedItem(item='hello', error_type='ValueError', message="invalid literal for int() with base 10: 'hello'"),
  RejectedItem(item='3.5', error_type='ValueError', message="invalid literal for int() with base 10: '3.5'")],
 [ValidationFailure(item=-2, reason='value must be non-negative')])

## Problem 9 — Instrument a Pipeline with Metrics

### Task

Create a metrics stage that tracks:

- number of received items;
- number of successfully forwarded items;
- number of downstream failures;
- most recent item.

The metrics object must remain inspectable after the pipeline is closed.

This problem demonstrates a useful principle: **state can live outside the generator**, making it easy to inspect without special coroutine commands.


In [20]:
@dataclass
class PipelineMetrics:
    received: int = 0
    forwarded: int = 0
    downstream_failures: int = 0
    last_item: Any = None


@coroutine
def metrics_stage(
    metrics: PipelineMetrics,
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Record forwarding metrics around a downstream target."""
    try:
        while True:
            item = yield
            metrics.received += 1
            metrics.last_item = item
            try:
                target.send(item)
            except Exception:
                metrics.downstream_failures += 1
                raise
            else:
                metrics.forwarded += 1
    finally:
        close_quietly(target)


In [21]:
metrics = PipelineMetrics()
metric_results: List[int] = []

problem9_pipeline = metrics_stage(
    metrics,
    map_stage(lambda value: value * 10, collector(metric_results)),
)

for value in [1, 2, 3, 4]:
    problem9_pipeline.send(value)

problem9_pipeline.close()

assert metric_results == [10, 20, 30, 40]
assert metrics == PipelineMetrics(
    received=4,
    forwarded=4,
    downstream_failures=0,
    last_item=4,
)

metrics


PipelineMetrics(received=4, forwarded=4, downstream_failures=0, last_item=4)

## Problem 10 — Dynamically Reconfigure a Transform Stage

### Task

Create a stateful stage whose transformation function can be changed while the pipeline is running.

Use explicit command objects rather than overloading ordinary data values.

Behavior:

- ordinary values are transformed and forwarded;
- `SetTransform(new_function)` changes the active transformation;
- command messages are not forwarded as data.


In [22]:
@dataclass(frozen=True)
class SetTransform:
    transform: Callable[[Any], Any]


@coroutine
def configurable_map_stage(
    initial_transform: Callable[[Any], Any],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    """Map values with a transform that can be replaced at runtime."""
    transform = initial_transform

    try:
        while True:
            message = yield

            if isinstance(message, SetTransform):
                transform = message.transform
                continue

            target.send(transform(message))
    finally:
        close_quietly(target)


In [23]:
reconfigured_results: List[int] = []
problem10_pipeline = configurable_map_stage(
    lambda value: value * 2,
    collector(reconfigured_results),
)

problem10_pipeline.send(3)  # 6
problem10_pipeline.send(5)  # 10
problem10_pipeline.send(SetTransform(lambda value: value ** 3))
problem10_pipeline.send(2)  # 8
problem10_pipeline.send(4)  # 64
problem10_pipeline.close()

assert reconfigured_results == [6, 10, 8, 64]
reconfigured_results


[6, 10, 8, 64]

## Problem 11 — Compose a Non-Trivial ETL Pipeline

### Scenario

A stream contains purchase records:

```python
{
    "customer": "alice",
    "category": "books",
    "amount": "19.99",
    "country": "BG"
}
```

Build one push pipeline that:

1. normalizes customer and category names to lowercase;
2. parses `amount` as `float`;
3. routes malformed records to a parse-error collector;
4. rejects non-positive amounts;
5. adds a 20% tax field;
6. broadcasts accepted records to:
   - an accepted-record collector;
   - a grouped-statistics branch keyed by category;
7. records pipeline metrics.

This problem combines multiple reusable stages developed earlier.


In [24]:
def normalize_purchase(record: Mapping[str, Any]) -> Dict[str, Any]:
    normalized = dict(record)
    normalized["customer"] = str(record["customer"]).strip().lower()
    normalized["category"] = str(record["category"]).strip().lower()
    normalized["country"] = str(record["country"]).strip().upper()
    normalized["amount"] = float(record["amount"])
    return normalized


def add_tax(record: Mapping[str, Any], rate: float = 0.20) -> Dict[str, Any]:
    enriched = dict(record)
    enriched["tax"] = round(enriched["amount"] * rate, 2)
    enriched["total_with_tax"] = round(enriched["amount"] + enriched["tax"], 2)
    return enriched


In [25]:
raw_purchases = [
    {"customer": " Alice ", "category": " Books ", "amount": "19.99", "country": "bg"},
    {"customer": "BOB", "category": "Games", "amount": "59.50", "country": "de"},
    {"customer": "Cara", "category": "Books", "amount": "-5", "country": "fr"},
    {"customer": "Dan", "category": "Music", "amount": "not-a-number", "country": "bg"},
    {"customer": "Eve", "category": "Books", "amount": "10", "country": "us"},
]

etl_parse_failures: List[RejectedItem] = []
etl_validation_failures: List[ValidationFailure] = []
accepted_purchases: List[Dict[str, Any]] = []
category_stats: List[GroupStats] = []
etl_metrics = PipelineMetrics()

category_stats_branch = map_stage(
    lambda record: (record["category"], record["total_with_tax"]),
    grouped_stats_stage(collector(category_stats)),
)

accepted_broadcast = broadcast(
    collector(accepted_purchases),
    category_stats_branch,
)

etl_pipeline = metrics_stage(
    etl_metrics,
    try_map_stage(
        normalize_purchase,
        validate_stage(
            predicate=lambda record: record["amount"] > 0,
            reason="amount must be positive",
            target=map_stage(add_tax, accepted_broadcast),
            rejected_target=collector(etl_validation_failures),
        ),
        error_target=collector(etl_parse_failures),
        handled_exceptions=(ValueError, TypeError, KeyError),
    ),
)

for purchase in raw_purchases:
    etl_pipeline.send(purchase)

etl_pipeline.close()

assert len(accepted_purchases) == 3
assert len(etl_parse_failures) == 1
assert len(etl_validation_failures) == 1
assert etl_metrics.received == 5
assert etl_metrics.forwarded == 5
assert category_stats[-1].key == "books"
assert category_stats[-1].count == 2

accepted_purchases, category_stats, etl_parse_failures, etl_validation_failures, etl_metrics


([{'customer': 'alice',
   'category': 'books',
   'amount': 19.99,
   'country': 'BG',
   'tax': 4.0,
   'total_with_tax': 23.99},
  {'customer': 'bob',
   'category': 'games',
   'amount': 59.5,
   'country': 'DE',
   'tax': 11.9,
   'total_with_tax': 71.4},
  {'customer': 'eve',
   'category': 'books',
   'amount': 10.0,
   'country': 'US',
   'tax': 2.0,
   'total_with_tax': 12.0}],
 [GroupStats(key='books', count=1, total=23.99, average=23.99),
  GroupStats(key='games', count=1, total=71.4, average=71.4),
  GroupStats(key='books', count=2, total=35.989999999999995, average=17.994999999999997)],
 [RejectedItem(item={'customer': 'Dan', 'category': 'Music', 'amount': 'not-a-number', 'country': 'bg'}, error_type='ValueError', message="could not convert string to float: 'not-a-number'")],
 [ValidationFailure(item={'customer': 'cara', 'category': 'books', 'amount': -5.0, 'country': 'FR'}, reason='amount must be positive')],
 PipelineMetrics(received=5, forwarded=5, downstream_failures=0

## Problem 12 — Build a Pipeline Factory

### Task

Long nested constructor expressions can become difficult to read.  
Create a factory function that builds a reusable numeric pipeline:

```text
input
  → parse float
  → keep values within configured bounds
  → apply a configured transform
  → batch results
  → collector
```

The factory must return:

- the input coroutine;
- the output list;
- the parse-error list;
- the validation-error list.

Use keyword-only configuration to make call sites self-documenting.


In [26]:
@dataclass
class NumericPipeline:
    input: Generator[Any, Any, Any]
    output_batches: List[List[float]]
    parse_errors: List[RejectedItem]
    validation_errors: List[ValidationFailure]


def build_numeric_pipeline(
    *,
    lower_bound: float,
    upper_bound: float,
    transform: Callable[[float], float],
    batch_size: int,
) -> NumericPipeline:
    if lower_bound > upper_bound:
        raise ValueError("lower_bound must not exceed upper_bound")

    output_batches: List[List[float]] = []
    parse_errors: List[RejectedItem] = []
    validation_errors: List[ValidationFailure] = []

    pipeline = try_map_stage(
        float,
        validate_stage(
            predicate=lambda value: lower_bound <= value <= upper_bound,
            reason=f"value must be within [{lower_bound}, {upper_bound}]",
            target=map_stage(
                transform,
                batch_stage(batch_size, collector(output_batches)),
            ),
            rejected_target=collector(validation_errors),
        ),
        error_target=collector(parse_errors),
        handled_exceptions=(ValueError, TypeError),
    )

    return NumericPipeline(
        input=pipeline,
        output_batches=output_batches,
        parse_errors=parse_errors,
        validation_errors=validation_errors,
    )


In [27]:
numeric = build_numeric_pipeline(
    lower_bound=0,
    upper_bound=10,
    transform=lambda value: round(value ** 2, 2),
    batch_size=2,
)

for raw_value in ["2", "4.5", "-1", "bad", "10", "7"]:
    numeric.input.send(raw_value)

numeric.input.close()

assert numeric.output_batches == [[4.0, 20.25], [100.0, 49.0]]
assert [item.item for item in numeric.parse_errors] == ["bad"]
assert [item.item for item in numeric.validation_errors] == [-1.0]

numeric


NumericPipeline(input=<generator object try_map_stage at 0x000002639AC87BE0>, output_batches=[[4.0, 20.25], [100.0, 49.0]], parse_errors=[RejectedItem(item='bad', error_type='ValueError', message="could not convert string to float: 'bad'")], validation_errors=[ValidationFailure(item=-1.0, reason='value must be within [0, 10]')])

# Additional Challenge Problems

Try these before reading the suggested solution sketches.

## Challenge A — Deduplication with Bounded Memory

Create a stage that removes duplicates seen within only the most recent `k` items.

Questions to consider:

- Is the window based on received items or forwarded items?
- How will you support unhashable values?
- What is the time complexity of removing old items?

## Challenge B — Rate-Limited Sink

Create a sink that allows at most `n` writes per second.

Questions to consider:

- A call to `time.sleep` blocks the entire synchronous pipeline. Is that acceptable?
- Would an `asyncio` pipeline be more appropriate?

## Challenge C — Join Two Logical Streams

Receive tagged messages such as:

```python
("customer", customer_id, customer_record)
("order", customer_id, order_record)
```

Buffer records until both sides are available, then emit joined records.

Questions to consider:

- How long can unmatched data remain in memory?
- How should duplicates be handled?
- What cleanup or expiration strategy is required?


## Suggested Solution — Bounded Recent Deduplication

This implementation defines the window over **received items**.  
It requires hashable keys but allows a `key` function to derive a hashable identity from complex objects.


In [28]:
@coroutine
def deduplicate_recent(
    capacity: int,
    target: Generator[Any, Any, Any],
    key: Callable[[Any], Hashable] = lambda item: item,
) -> Generator[None, Any, None]:
    """Suppress items whose key appeared among the previous capacity inputs."""
    if capacity <= 0:
        raise ValueError("capacity must be positive")

    recent: Deque[Hashable] = deque()
    counts: Dict[Hashable, int] = defaultdict(int)

    try:
        while True:
            item = yield
            item_key = key(item)

            is_duplicate = counts[item_key] > 0

            recent.append(item_key)
            counts[item_key] += 1

            if len(recent) > capacity:
                expired_key = recent.popleft()
                counts[expired_key] -= 1
                if counts[expired_key] == 0:
                    del counts[expired_key]

            if not is_duplicate:
                target.send(item)
    finally:
        close_quietly(target)


In [29]:
deduplicated: List[str] = []
dedupe_pipeline = deduplicate_recent(3, collector(deduplicated))

for item in ["A", "B", "A", "C", "D", "E", "A", "B"]:
    dedupe_pipeline.send(item)

dedupe_pipeline.close()

# "A" is suppressed at position 3 because it is still recent.
# It is accepted later after enough newer inputs make the earlier "A" expire.
assert deduplicated == ["A", "B", "C", "D", "E", "A", "B"]
deduplicated


['A', 'B', 'C', 'D', 'E', 'A', 'B']

# Testing Strategies

For production-quality pipeline code, test more than the happy path.

Recommended test categories:

1. **Empty input** — closing a pipeline before sending values should not fail.
2. **Single item** — especially important for batching and rolling windows.
3. **Boundary values** — exactly equal to lower and upper limits.
4. **Malformed data** — verify routing and error metadata.
5. **Close behavior** — buffered data should flush exactly once.
6. **Downstream failure** — confirm whether the failure propagates or is intentionally handled.
7. **Aliasing** — downstream mutation should not corrupt internal buffers.
8. **Repeated close** — generator `.close()` should remain harmless.
9. **Control-message collisions** — unique sentinel objects avoid collisions with ordinary values.
10. **Ordering** — synchronous pipelines should preserve input order unless a stage deliberately changes it.


In [30]:
def run_regression_checks() -> None:
    # Empty close
    empty_batches: List[List[int]] = []
    empty_pipeline = batch_stage(2, collector(empty_batches))
    empty_pipeline.close()
    assert empty_batches == []

    # Exact boundary values
    boundaries: List[float] = []
    boundary_pipeline = filter_stage(
        lambda value: 0 <= value <= 10,
        collector(boundaries),
    )
    for value in [-1, 0, 10, 11]:
        boundary_pipeline.send(value)
    boundary_pipeline.close()
    assert boundaries == [0, 10]

    # Buffer copying / aliasing
    copied_batches: List[List[int]] = []
    copy_pipeline = batch_stage(2, collector(copied_batches))
    copy_pipeline.send(1)
    copy_pipeline.send(2)
    first_batch = copied_batches[0]
    first_batch.append(999)

    copy_pipeline.send(3)
    copy_pipeline.send(4)
    copy_pipeline.close()

    assert copied_batches[1] == [3, 4]

    # Repeated close should be harmless.
    copy_pipeline.close()


run_regression_checks()
print("All regression checks passed.")


All regression checks passed.


# Performance and Architectural Notes

## Complexity

For a linear pipeline with `s` stages and `n` items, the usual runtime is approximately:

```text
O(n × s)
```

Stateful stages add their own costs:

- rolling average with `deque`: `O(1)` update plus average calculation;
- routing by dictionary key: expected `O(1)` lookup;
- recent deduplication: expected `O(1)` membership and expiration;
- grouped aggregation: expected `O(1)` update per item.

## Synchronous behavior

Generator-coroutine pipelines are synchronous:

- `.send()` does not return until all downstream processing for that item finishes;
- one slow stage blocks the entire chain;
- fan-out branches are processed sequentially;
- there is no built-in queue, parallelism, or backpressure protocol.

For CPU-heavy parallel work, multi-process designs may be more appropriate.  
For high-concurrency I/O, consider `asyncio`, queues, streams, or a dedicated event-processing framework.

## Ownership and shutdown

This notebook uses the convention that a stage **owns** and closes its downstream targets.  
In a larger system, ownership must be documented carefully, especially when targets are shared by multiple upstream stages.

## Error policy

Choose one policy explicitly:

- fail fast and propagate;
- retry;
- skip;
- route to a dead-letter queue;
- replace with a default;
- stop only one branch.

Silently swallowing unexpected exceptions makes pipelines difficult to debug.


# Final Capstone

Design a push pipeline for log analytics with the following requirements:

1. parse JSON log lines;
2. route invalid JSON to a dead-letter queue;
3. normalize timestamps;
4. discard health-check endpoints;
5. route records by HTTP status class (`2xx`, `4xx`, `5xx`);
6. batch each route independently;
7. compute per-endpoint counts;
8. emit metrics every 1,000 received records;
9. flush every partial batch on shutdown;
10. document the error and ownership policies.

A strong solution should reuse generic stages, keep parsing separate from validation, and test shutdown behavior explicitly.


## Summary

You now have reusable patterns for advanced push-based coroutine pipelines:

- `map_stage`
- `filter_stage`
- `tap_stage`
- `try_map_stage`
- `validate_stage`
- `broadcast`
- `route_by_key`
- `batch_stage`
- `rolling_average_stage`
- `grouped_stats_stage`
- `metrics_stage`
- `configurable_map_stage`
- `deduplicate_recent`

The central design idea is simple:

```text
receive with yield → process → push with target.send(...)
```

The engineering quality comes from explicit ownership, cleanup, error routing, testing, and clear semantics for stateful stages.
